# GRU
- gru, derin öğrenme alanında kullanılan , özellikle sırlaı verieler üzerinde çalışan bir rnn türüdür
- lstm ile benzer işlevlere sahip olsa da , daha basit yapıya sahiptir
-lstm e göre daha basit yapı:  daha az parametreye sahip oludüüu için daha hızlı eğitilir
- uzun vadeli bağımlılıkları yakalama: lstm gibi uzun vadeli bağımlılıklaı öprenme konusunda başarılıdırüü
- gradyan ...
- hru iki geriş kapısı vardır: güncelleme ve sıfırlama kapıları
- sıfıtlsms geçidi: bu geçit, önceki gizli durumun ne jadarını yeni bilfgilere göre gğncelleyeceine karar verr
- güncelleme kapısı: yeni bilgilerin ne kadarının mevcut gizli duruma eklenip eklenmeyeceğine karar verir
- bu ikigeçit sayesinde  , gru lar hangi ilgileri hatırlayacakalrına ve hangi bilgileri unutacaklarına dah iyi karar verirler
- kullanı  alanları : doğal dil işleme, zaman serisi analizi, konuşma yanıma ve diğer sıralı veri uygulamaları
- lstm ile;
- bezerlikler: ikisi de sıralı veriler üzeride çalışır, uzun vadeli bağımlılıkları yakalayabilir
- farklılıklar: gru , lstm e göre daha az parametre için hızlı. ancak duruma göre değişir
- özetle,  gru sıralı veriler üzerinde çalışan ve uzun vadeli bağımlılıkları yakalayabilen güçlü bir derin öğrenme modelidir. lstm e göre daha basit yapıya sahiptir ve daha hızlı eğitilir. doğal dil işleme, zaman serisi analizi ve diğer sıralı veri uygulamalarında yaygın olarak kullanılır

In [ ]:
#GRU
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense
from sklearn.preprocessing import MinMaxScaler
import pickle

# Veriyi yükle
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
data = pd.read_csv(url, usecols=[1])
data = data.values.astype('float32')

# Veriyi 0 ile 1 arasında ölçekleyelim
scaler = MinMaxScaler(feature_range=(0, 1))
dataset = scaler.fit_transform(data)

# Veriyi eğitim ve test olarak ayırma (80% eğitim, 20% test)
train_size = int(len(dataset) * 0.8)
test_size = len(dataset) - train_size
train, test = dataset[0:train_size, :], dataset[train_size:len(dataset), :]

# GRU modeli için veri şekillendirme
def create_dataset(dataset, time_step=1):
    dataX, dataY = [], []
    for i in range(len(dataset) - time_step - 1):
        a = dataset[i:(i + time_step), 0]
        dataX.append(a)
        dataY.append(dataset[i + time_step, 0])
    return np.array(dataX), np.array(dataY)

time_step = 10
X_train, y_train = create_dataset(train, time_step)
X_test, y_test = create_dataset(test, time_step)

# Veriyi GRU girişine uygun hale getirme
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

# GRU modeli oluşturma
model = Sequential()
model.add(GRU(50, return_sequences=True, input_shape=(time_step, 1)))
model.add(GRU(50, return_sequences=False))
model.add(Dense(1))

model.compile(optimizer='adam', loss='mean_squared_error')

# Modeli eğitme
history = model.fit(X_train, y_train, epochs=20, batch_size=1, validation_data=(X_test, y_test))

# Tahminler
train_predict = model.predict(X_train)
test_predict = model.predict(X_test)

# Tahminleri ölçeklendirmeyi geri alma
train_predict = scaler.inverse_transform(train_predict)
test_predict = scaler.inverse_transform(test_predict)

# Gerçek değerleri ölçeklendirmeyi geri alma
y_train_actual = scaler.inverse_transform(y_train.reshape(-1, 1))
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))

# Eğitim ve test verisi için tahminleri görselleştirme
plt.figure(figsize=(12, 6))
plt.plot(data, label='Actual Data')
plt.plot(range(time_step, len(train_predict) + time_step), train_predict, label='Train Predict')
plt.plot(range(len(train_predict) + (time_step * 2), len(train_predict) + (time_step * 2) + len(test_predict)), test_predict, label='Test Predict')
plt.xlabel('Time')
plt.ylabel('Passengers')
plt.legend()
plt.show()

# Modeli kaydetme
model.save('air_passenger_gru_model.h5')

from sklearn.metrics import mean_absolute_error, mean_squared_error

# MAE, MSE ve RMSE hesaplama
train_mae = mean_absolute_error(y_train_actual, train_predict)
test_mae = mean_absolute_error(y_test_actual, test_predict)
train_mse = mean_squared_error(y_train_actual, train_predict)
test_mse = mean_squared_error(y_test_actual, test_predict)
train_rmse = np.sqrt(train_mse)
test_rmse = np.sqrt(test_mse)

print(f'Train MAE: {train_mae:.2f}')
print(f'Test MAE: {test_mae:.2f}')
print(f'Train MSE: {train_mse:.2f}')
print(f'Test MSE: {test_mse:.2f}')
print(f'Train RMSE: {train_rmse:.2f}')
print(f'Test RMSE: {test_rmse:.2f}')